In [1]:
import os
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback

from week5_trading_env_v3 import TradingEnvV3


class EpisodeRewardLogger(BaseCallback):

    def __init__(self):

        super().__init__()

        self.episode_rewards = []

        self.current_reward = 0.0

    def _on_step(self):

        reward = self.locals["rewards"][0]

        done = self.locals["dones"][0]

        self.current_reward += reward

        if done:

            self.episode_rewards.append(
                self.current_reward
            )

            self.current_reward = 0.0

        return True


def train_agent():

    env = TradingEnvV3(
        rho=0.8
    )

    logger = EpisodeRewardLogger()

    model = PPO(

        "MlpPolicy",

        env,

        learning_rate=3e-4,

        n_steps=2048,

        batch_size=64,

        gamma=0.99,

        verbose=1

    )

    model.learn(

        total_timesteps=50000,

        callback=logger

    )

    os.makedirs(
        "models",
        exist_ok=True
    )

    model.save(
        "models/week5_ppo"
    )

    env.close()

    return model, logger.episode_rewards


def evaluate_agent(

    model,

    episodes=10

):

    env = TradingEnvV3(
        rho=0.8
    )

    rewards = []

    pnl = []

    for ep in range(episodes):

        obs, info = env.reset()

        done = False

        total_reward = 0.0

        running = 0.0

        curve = []

        while not done:

            action, _ = model.predict(

                obs,

                deterministic=True

            )

            obs, reward, terminated, truncated, info = env.step(action)

            total_reward += reward

            running += reward

            curve.append(running)

            done = terminated or truncated

        rewards.append(total_reward)

        pnl.append(curve)

        print(

            f"Episode {ep+1:2d} "

            f"Reward = {total_reward:.2f}"

        )

    env.close()

    return rewards, pnl


def random_agent(

    episodes=10

):

    env = TradingEnvV3(
        rho=0.8
    )

    rewards = []

    for ep in range(episodes):

        obs, info = env.reset()

        done = False

        total_reward = 0.0

        while not done:

            action = env.action_space.sample()

            obs, reward, terminated, truncated, info = env.step(action)

            total_reward += reward

            done = terminated or truncated

        rewards.append(total_reward)

    env.close()

    return rewards

def plot_training_rewards(rewards):

    os.makedirs(
        "plots",
        exist_ok=True
    )

    rewards = np.array(
        rewards
    )

    plt.figure(
        figsize=(8,4)
    )

    plt.plot(
        rewards,
        alpha=0.4,
        label="Episode Reward"
    )

    if len(rewards) >= 10:

        moving = np.convolve(
            rewards,
            np.ones(10)/10,
            mode="valid"
        )

        plt.plot(
            moving,
            linewidth=2,
            label="Moving Average"
        )

    plt.xlabel(
        "Episode"
    )

    plt.ylabel(
        "Reward"
    )

    plt.title(
        "Week 5 PPO Training"
    )

    plt.legend()

    plt.tight_layout()

    plt.savefig(
        "plots/week5_training_rewards.png"
    )

    plt.close()


def plot_pnl(pnl):

    plt.figure(
        figsize=(8,4)
    )

    for curve in pnl:

        plt.plot(
            curve,
            alpha=0.5
        )

    plt.xlabel(
        "Step"
    )

    plt.ylabel(
        "Cumulative Reward"
    )

    plt.title(
        "Evaluation Episodes"
    )

    plt.tight_layout()

    plt.savefig(
        "plots/week5_cumulative_pnl.png"
    )

    plt.close()


if __name__ == "__main__":

    model, training_rewards = train_agent()

    plot_training_rewards(
        training_rewards
    )

    eval_rewards, pnl = evaluate_agent(
        model,
        episodes=10
    )

    random_rewards = random_agent(
        episodes=10
    )

    plot_pnl(
        pnl
    )

    print()

    print(
        "PPO Mean Reward :",
        round(
            np.mean(eval_rewards),
            2
        )
    )

    print(
        "Random Mean Reward :",
        round(
            np.mean(random_rewards),
            2
        )
    )

    print()

    print(
        "Best PPO Reward :",
        round(
            np.max(eval_rewards),
            2
        )
    )

    print(
        "Worst PPO Reward :",
        round(
            np.min(eval_rewards),
            2
        )
    )

    print()

    print(
        "Training plot saved to plots/week5_training_rewards.png"
    )

    print(
        "PnL plot saved to plots/week5_cumulative_pnl.png"
    )

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | -9.31    |
| time/              |          |
|    fps             | 500      |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 100         |
|    ep_rew_mean          | -6.23       |
| time/                   |             |
|    fps                  | 349         |
|    iterations           | 2           |
|    time_elapsed         | 11          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009986188 |
|    clip_fraction        | 0.029       |
|    clip_range           | 0.2         |
|    entropy_loss   